# CardioIA — Fase 4 — Parte 2: CNN, Transfer Learning e Protótipo

**Pré-requisito:** Parte 1 executada — `camus_splits.csv` no Drive (`dataset_cardio/`).

**Tarefa:** classificar ecocardiogramas (CAMUS) pela **estrutura cardíaca predominante** (`estrutura_253`, `estrutura_254`, `estrutura_255`).

### Entregáveis desta Parte

1. **CNN simples** treinada do zero
2. **Transfer learning** (VGG16)
3. Métricas: acurácia, matriz de confusão, precisão, recall, F1
4. **Protótipo:** upload de imagem → predição (somente a imagem, sem máscara)

### Colab

1. **Runtime → Change runtime type → GPU** (recomendado)
2. Execute as células de cima para baixo

**Vídeo / apresentação (sem retreinar):** na célula de configuração, deixe `MODO_APRESENTACAO = True`. O notebook carrega os modelos `.keras` do Drive e **pula o treino**. Execute até a célula **"Carregar modelos salvos"** antes do protótipo.

> Protótipo acadêmico — simulação educacional; não substitui diagnóstico médico.

### Dependências

In [ ]:
!pip install -q pandas numpy matplotlib seaborn pillow scikit-learn opencv-python-headless tensorflow

### Montar Drive e carregar splits (Parte 1)

In [ ]:
import json
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import tensorflow as tf
from google.colab import drive
from PIL import Image
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
)
from sklearn.utils.class_weight import compute_class_weight
from tensorflow.keras import layers, models
from tensorflow.keras.applications.vgg16 import VGG16, preprocess_input
from tensorflow.keras.callbacks import EarlyStopping

drive.mount("/content/drive")

CAMINHO_BASE = Path("/content/drive/MyDrive/dataset_cardio")
ARQUIVO_SPLITS = CAMINHO_BASE / "camus_splits.csv"
ARQUIVO_CONFIG = CAMINHO_BASE / "camus_preprocess_config.json"
PASTA_EVIDENCIAS = CAMINHO_BASE / "evidencias_fase4"
PASTA_EVIDENCIAS.mkdir(parents=True, exist_ok=True)

SEED = 42
IMG_SIZE = (224, 224)
BATCH_SIZE = 32
EPOCHS_CNN = 20
EPOCHS_TL = 15

# True = carrega modelos do Drive, pula treino
# False = execução completa (primeira vez ou retreino)
MODO_APRESENTACAO = True

tf.random.set_seed(SEED)
np.random.seed(SEED)

df_splits = pd.read_csv(ARQUIVO_SPLITS, encoding="utf-8")
if ARQUIVO_CONFIG.exists():
    config = json.loads(ARQUIVO_CONFIG.read_text(encoding="utf-8"))
    print("Config Parte 1:", config.get("tarefa"), "| classes:", config.get("classes"))

print("Splits:", df_splits["split"].value_counts().to_dict())
print("Arquivo:", ARQUIVO_SPLITS)

### Pré-processamento (mesmo pipeline da Parte 1)

CNN simples usa pixels **[0, 1]**. VGG16 usa **`preprocess_input`** do ImageNet.

In [ ]:
def preprocessar_basico(caminho: str | Path) -> np.ndarray:
    with Image.open(caminho) as img:
        arr = np.array(img.convert("RGB"))
    arr = cv2.resize(arr, IMG_SIZE, interpolation=cv2.INTER_AREA)
    return (arr.astype(np.float32) / 255.0)


def preprocessar_vgg(caminho: str | Path) -> np.ndarray:
    with Image.open(caminho) as img:
        arr = np.array(img.convert("RGB"))
    arr = cv2.resize(arr, IMG_SIZE, interpolation=cv2.INTER_AREA)
    return preprocess_input(arr.astype(np.float32))


NOMES_CLASSE = sorted(df_splits["classe"].unique())
NUM_CLASSES = len(NOMES_CLASSE)
label_para_idx = {nome: i for i, nome in enumerate(NOMES_CLASSE)}
idx_para_label = {i: nome for nome, i in label_para_idx.items()}


def carregar_split(nome_split: str, preprocess_fn) -> tuple[np.ndarray, np.ndarray]:
    parte = df_splits[df_splits["split"] == nome_split]
    X, y = [], []
    for _, row in parte.iterrows():
        X.append(preprocess_fn(row["caminho_frame"]))
        y.append(label_para_idx[row["classe"]])
    return np.array(X), np.array(y)


print("Classes:", NOMES_CLASSE)

if MODO_APRESENTACAO:
    print("Modo apresentação: carregando apenas conjunto de teste (~30 s)…")
    X_test, y_test = carregar_split("teste", preprocessar_basico)
    X_test_vgg, _ = carregar_split("teste", preprocessar_vgg)
    print(f"Teste {X_test.shape}")
else:
    print("Carregando imagens (pode levar 1–2 min)…")
    X_train, y_train = carregar_split("treino", preprocessar_basico)
    X_val, y_val = carregar_split("validacao", preprocessar_basico)
    X_test, y_test = carregar_split("teste", preprocessar_basico)

    X_train_vgg, _ = carregar_split("treino", preprocessar_vgg)
    X_val_vgg, _ = carregar_split("validacao", preprocessar_vgg)
    X_test_vgg, _ = carregar_split("teste", preprocessar_vgg)

    y_train_oh = tf.keras.utils.to_categorical(y_train, NUM_CLASSES)
    y_val_oh = tf.keras.utils.to_categorical(y_val, NUM_CLASSES)

    pesos = compute_class_weight("balanced", classes=np.arange(NUM_CLASSES), y=y_train)
    class_weight = {i: pesos[i] for i in range(NUM_CLASSES)}
    print("Pesos por classe (desbalanceamento):", class_weight)
    print(f"Treino {X_train.shape} | Val {X_val.shape} | Teste {X_test.shape}")

### Interpretação assistida (simulação educacional)

Mapeamento das classes técnicas (`estrutura_253/254/255`) para **linguagem clínica compreensível**, com aviso explícito de que **não constitui diagnóstico** — apenas apoio didático ao protótipo CardioIA.

In [ ]:
# Mapeamento: pixels CAMUS (Kaggle) → regiões anotadas no dataset original
MAPA_INTERPRETACAO = {
    "estrutura_253": {
        "nome_legivel": "Átrio esquerdo (região predominante)",
        "descricao_tecnica": "Classe em que o pixel 253 domina a máscara de segmentação expert.",
        "contexto_simulado": (
            "Padrão visual associado à predominância do átrio esquerdo na segmentação CAMUS. "
            "Em laudo real, o cardiologista avaliaria volume atrial e função diastólica — "
            "aqui o modelo apenas classifica a região anatômica predominante."
        ),
    },
    "estrutura_254": {
        "nome_legivel": "Cavidade do ventrículo esquerdo (região predominante)",
        "descricao_tecnica": "Classe em que o pixel 254 domina a máscara de segmentação expert.",
        "contexto_simulado": (
            "Padrão visual associado à predominância da cavidade do ventrículo esquerdo. "
            "Em contexto clínico, essa região é central para avaliar volume e função sistólica — "
            "o protótipo **não** calcula fração de ejeção nem conclui normalidade."
        ),
    },
    "estrutura_255": {
        "nome_legivel": "Miocárdio ventricular (região predominante)",
        "descricao_tecnica": "Classe em que o pixel 255 domina a máscara de segmentação expert.",
        "contexto_simulado": (
            "Padrão visual associado à predominância do miocárdio ventricular na segmentação. "
            "Espessura e movimentação parietal seriam analisadas pelo especialista — "
            "o modelo não detecta doença estrutural."
        ),
    },
}

AVISO_MEDICO = """
╔══════════════════════════════════════════════════════════════════╗
║  AVISO IMPORTANTE — PROTÓTIPO ACADÊMICO CARDIOIA                 ║
╠══════════════════════════════════════════════════════════════════╣
║  • Isto NÃO é diagnóstico médico nem laudo de ecocardiograma.    ║
║  • NÃO indica presença ou ausência de doença cardíaca.           ║
║  • NÃO substitui consulta, exame presencial ou cardiologista.    ║
║  • Qualquer decisão de saúde exige avaliação de profissional     ║
║    habilitado (médico / cardiologista) com histórico clínico.    ║
╚══════════════════════════════════════════════════════════════════╝
"""

ROTULOS_GRAFICO = [MAPA_INTERPRETACAO[c]["nome_legivel"] for c in NOMES_CLASSE]


def gerar_parecer(classe: str, confianca: float) -> str:
    info = MAPA_INTERPRETACAO[classe]
    confianca_txt = "alta" if confianca >= 0.75 else ("moderada" if confianca >= 0.5 else "baixa")
    return f"""
── Interpretação assistida (simulação) ──
Região identificada: {info['nome_legivel']}
Confiança do modelo: {confianca:.1%} ({confianca_txt})

O que isso significa (escopo educacional):
{info['contexto_simulado']}

Recomendação:
Procure um médico ou cardiologista para interpretação clínica completa do exame,
correlação com sintomas e conduta terapêutica, se necessário.
{AVISO_MEDICO}
"""

### Carregar modelos salvos (modo apresentação)

**Obrigatório** quando `MODO_APRESENTACAO = True` — substitui as células de treino. Modelos em `dataset_cardio/evidencias_fase4/*.keras`.

In [ ]:
def carregar_modelos_salvos():
    caminho_cnn = PASTA_EVIDENCIAS / "modelo_cnn_simples.keras"
    caminho_vgg = PASTA_EVIDENCIAS / "modelo_vgg16_tl.keras"
    if not caminho_cnn.exists() or not caminho_vgg.exists():
        raise FileNotFoundError(
            f"Modelos não encontrados em {PASTA_EVIDENCIAS}. "
            "Defina MODO_APRESENTACAO = False e execute o treino completo uma vez."
        )
    cnn = tf.keras.models.load_model(caminho_cnn)
    modelo_vgg = tf.keras.models.load_model(caminho_vgg)
    print("Modelos carregados:", caminho_cnn.name, "|", caminho_vgg.name)
    return cnn, modelo_vgg


if MODO_APRESENTACAO:
    cnn, modelo_vgg = carregar_modelos_salvos()
else:
    print("Modo treino: modelos serão criados nas células abaixo.")

### Abordagem A — CNN simples do zero

In [ ]:
def criar_cnn_simples() -> tf.keras.Model:
    return models.Sequential(
        [
            layers.Input(shape=(*IMG_SIZE, 3)),
            layers.Conv2D(32, 3, activation="relu", padding="same"),
            layers.MaxPooling2D(),
            layers.Conv2D(64, 3, activation="relu", padding="same"),
            layers.MaxPooling2D(),
            layers.Conv2D(128, 3, activation="relu", padding="same"),
            layers.MaxPooling2D(),
            layers.Flatten(),
            layers.Dropout(0.3),
            layers.Dense(128, activation="relu"),
            layers.Dense(NUM_CLASSES, activation="softmax"),
        ],
        name="cnn_simples",
    )


if MODO_APRESENTACAO:
    print("Treino CNN ignorado — usando modelo carregado do Drive.")
else:
    cnn = criar_cnn_simples()
    cnn.compile(
        optimizer="adam",
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"],
    )
    cnn.summary()

    historico_cnn = cnn.fit(
        X_train,
        y_train,
        validation_data=(X_val, y_val),
        epochs=EPOCHS_CNN,
        batch_size=BATCH_SIZE,
        class_weight=class_weight,
        callbacks=[EarlyStopping(monitor="val_loss", patience=4, restore_best_weights=True)],
        verbose=1,
    )

### Abordagem B — Transfer Learning (VGG16)

In [ ]:
if MODO_APRESENTACAO:
    print("Treino VGG16 ignorado — usando modelo carregado do Drive.")
else:
    base_vgg = VGG16(weights="imagenet", include_top=False, input_shape=(*IMG_SIZE, 3))
    base_vgg.trainable = False

    entrada = layers.Input(shape=(*IMG_SIZE, 3))
    x = base_vgg(entrada, training=False)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(0.3)(x)
    x = layers.Dense(128, activation="relu")(x)
    saida = layers.Dense(NUM_CLASSES, activation="softmax")(x)
    modelo_vgg = models.Model(entrada, saida, name="vgg16_transfer")

    modelo_vgg.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"],
    )
    modelo_vgg.summary()

    historico_vgg = modelo_vgg.fit(
        X_train_vgg,
        y_train,
        validation_data=(X_val_vgg, y_val),
        epochs=EPOCHS_TL,
        batch_size=BATCH_SIZE,
        class_weight=class_weight,
        callbacks=[EarlyStopping(monitor="val_loss", patience=3, restore_best_weights=True)],
        verbose=1,
    )

### Curvas de treino

In [ ]:
def plotar_historico(hist, titulo: str, arquivo: str) -> None:
    fig, axes = plt.subplots(1, 2, figsize=(10, 4))
    axes[0].plot(hist.history["loss"], label="treino")
    axes[0].plot(hist.history["val_loss"], label="validação")
    axes[0].set_title(f"{titulo} — loss")
    axes[0].legend()
    axes[1].plot(hist.history["accuracy"], label="treino")
    axes[1].plot(hist.history["val_accuracy"], label="validação")
    axes[1].set_title(f"{titulo} — acurácia")
    axes[1].legend()
    plt.tight_layout()
    caminho = PASTA_EVIDENCIAS / arquivo
    plt.savefig(caminho, dpi=120, bbox_inches="tight")
    plt.show()
    print("Salvo:", caminho)


if MODO_APRESENTACAO:
    for arquivo in ("curvas_cnn_simples.png", "curvas_vgg16.png"):
        caminho = PASTA_EVIDENCIAS / arquivo
        if caminho.exists():
            display(Image.open(caminho))
            print("Curvas já salvas:", caminho)
        else:
            print("Pule curvas ou use prints em assets/fase4/ —", caminho.name, "não encontrado.")
else:
    plotar_historico(historico_cnn, "CNN simples", "curvas_cnn_simples.png")
    plotar_historico(historico_vgg, "VGG16 transfer learning", "curvas_vgg16.png")

### Avaliação no conjunto de teste

Métricas exigidas pelo enunciado: acurácia, matriz de confusão, precisão, recall, F1-score.

In [ ]:
def avaliar(modelo, X, y_true, nome: str, arquivo_matriz: str) -> dict:
    y_pred = np.argmax(modelo.predict(X, verbose=0), axis=1)
    metricas = {
        "modelo": nome,
        "acuracia": accuracy_score(y_true, y_pred),
        "precisao_macro": precision_score(y_true, y_pred, average="macro", zero_division=0),
        "recall_macro": recall_score(y_true, y_pred, average="macro", zero_division=0),
        "f1_macro": f1_score(y_true, y_pred, average="macro", zero_division=0),
    }
    print(f"\n=== {nome} ===")
    print(f"Acurácia: {metricas['acuracia']:.4f}")
    print(classification_report(y_true, y_pred, target_names=NOMES_CLASSE, zero_division=0))

    cm = confusion_matrix(y_true, y_pred)
    fig, ax = plt.subplots(figsize=(6, 5))
    sns.heatmap(
        cm,
        annot=True,
        fmt="d",
        cmap="Blues",
        xticklabels=NOMES_CLASSE,
        yticklabels=NOMES_CLASSE,
        ax=ax,
    )
    ax.set_xlabel("Predito")
    ax.set_ylabel("Real")
    ax.set_title(f"Matriz de confusão — {nome}")
    plt.tight_layout()
    caminho = PASTA_EVIDENCIAS / arquivo_matriz
    plt.savefig(caminho, dpi=120, bbox_inches="tight")
    plt.show()
    print("Salvo:", caminho)
    return metricas


resultado_cnn = avaliar(cnn, X_test, y_test, "CNN simples", "matriz_confusao_cnn_simples.png")
resultado_vgg = avaliar(modelo_vgg, X_test_vgg, y_test, "VGG16 TL", "matriz_confusao_vgg16.png")

comparacao = pd.DataFrame([resultado_cnn, resultado_vgg])
display(comparacao)
comparacao.to_csv(PASTA_EVIDENCIAS / "comparacao_metricas.csv", index=False, encoding="utf-8")

### Salvar modelos (Drive)

In [ ]:
if MODO_APRESENTACAO:
    print("Modelos já estão no Drive — salvamento ignorado.")
else:
    cnn.save(PASTA_EVIDENCIAS / "modelo_cnn_simples.keras")
    modelo_vgg.save(PASTA_EVIDENCIAS / "modelo_vgg16_tl.keras")
    print("Modelos salvos em", PASTA_EVIDENCIAS)

### Protótipo — Assistente Cardiológico Virtual (upload)

Simula o assistente do enunciado: envie um ecocardiograma e receba **interpretação assistida simulada** + aviso médico. Usa o **VGG16** treinado (troque para `modelo_vgg` → `cnn` se preferir).

In [ ]:
from google.colab import files
from IPython.display import HTML, display

# Garante modelos carregados se você pulou a célula de carregamento
if "modelo_vgg" not in globals():
    cnn, modelo_vgg = carregar_modelos_salvos()

print("CardioIA — Assistente Cardiológico Virtual (protótipo acadêmico)\n")
display(HTML(
    "<div style='background:#fff3cd;border:1px solid #ffc107;padding:12px;border-radius:8px;'>"
    "<b>⚠ Aviso:</b> simulação educacional FIAP. <b>Não</b> substitui cardiologista ou "
    "intervenção médica. Em caso de sintomas (dor no peito, falta de ar), procure "
    "<b>atendimento de urgência</b>."
    "</div>"
))

uploaded = files.upload()

if uploaded:
    nome = next(iter(uploaded))
    caminho_temp = Path("/content") / nome
    caminho_temp.write_bytes(uploaded[nome])

    img_vgg = preprocessar_vgg(caminho_temp)
    probs = modelo_vgg.predict(img_vgg[np.newaxis, ...], verbose=0)[0]
    idx = int(np.argmax(probs))
    classe = NOMES_CLASSE[idx]
    confianca = float(probs[idx])

    fig, axes = plt.subplots(1, 2, figsize=(11, 4))
    axes[0].imshow(np.array(Image.open(caminho_temp).convert("RGB")))
    axes[0].set_title("Ecocardiograma enviado")
    axes[0].axis("off")
    cores = ["#4c72b0" if i == idx else "#cccccc" for i in range(NUM_CLASSES)]
    axes[1].bar(ROTULOS_GRAFICO, probs, color=cores)
    axes[1].set_ylim(0, 1)
    axes[1].set_title(
        f"Predição: {MAPA_INTERPRETACAO[classe]['nome_legivel']}\n({confianca:.1%})",
        fontsize=10,
    )
    axes[1].tick_params(axis="x", rotation=12, labelsize=8)
    plt.tight_layout()
    plt.show()

    print(gerar_parecer(classe, confianca))
    print("Probabilidades por classe:")
    for c, p in zip(NOMES_CLASSE, probs):
        print(f"  • {MAPA_INTERPRETACAO[c]['nome_legivel']}: {p:.1%}")

### Conclusão (Parte 2)

- **CNN simples** e **VGG16 (transfer learning)** treinados nos splits da Parte 1.
- Métricas e matrizes salvas em `dataset_cardio/evidencias_fase4/` (copie prints para `assets/fase4/` no repositório, se desejar).
- Protótipo de upload demonstra inferência **apenas com a imagem**.

**Relatório Parte 2:** comparar modelos, interpretar desempenho na classe minoritária (`estrutura_253`) e registrar limitações éticas.